In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

In [2]:
df = pd.read_csv("processed_data.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

## Statistics

In [14]:
# --- 2.1 DESCRIPTIVE STATISTICS ---
print("=== 2.1 DESCRIPTIVE STATISTICS TABLE ===")
stats_summary = df[
    [
        "Accel_X_Smooth",
        "Accel_Y_Smooth",
        "Accel_Z_Smooth",
        "Accel_Mag_Smooth",
        "PIR_Motion",
    ]
].describe(percentiles=[0.25, 0.50, 0.75, 0.95])
print(stats_summary.round(4))

print("\n=== CORRELATION MATRIX ===")
corr_matrix = df[
    ["Accel_X_Smooth", "Accel_Y_Smooth", "Accel_Z_Smooth", "Accel_Mag_Smooth", "PIR_Motion"]
].corr()
print(corr_matrix.round(4))

=== 2.1 DESCRIPTIVE STATISTICS TABLE ===
       Accel_X_Smooth  Accel_Y_Smooth  Accel_Z_Smooth  Accel_Mag_Smooth  \
count      25895.0000      25895.0000      25895.0000        25895.0000   
mean          -0.0121         -0.0210          1.0137            0.0647   
std            0.1039          0.0549          0.0170            0.1043   
min           -0.7012         -1.0040          0.6870            0.0203   
25%            0.0160         -0.0056          1.0184            0.0254   
50%            0.0160         -0.0050          1.0190            0.0257   
75%            0.0174         -0.0022          1.0190            0.0276   
95%            0.0206         -0.0008          1.0198            0.2526   
max            3.2128          0.4120          1.1158            3.2206   

       PIR_Motion  
count  25895.0000  
mean       0.0404  
std        0.1969  
min        0.0000  
25%        0.0000  
50%        0.0000  
75%        0.0000  
95%        0.0000  
max        1.0000  

=== COR

## Visualisation

In [4]:
# show Accelerometer and PIR chart 
fig1 = make_subplots(specs=[[{"secondary_y": True}]])

fig1.add_trace(
    go.Scatter(
        x=df["elapsed_seconds"],
        y=df["Accel_Mag_Smooth"],
        name="Accel Mag (g)",
        line=dict(color="blue", width=1.5),
    ),
    secondary_y=False,
)
fig1.add_trace(
    go.Scatter(
        x=df["elapsed_seconds"],
        y=df["PIR_Motion"],
        name="PIR Motion Flag",
        line=dict(color="red", width=1, dash="dot"),
    ),
    secondary_y=True,
)

fig1.update_layout(
    title="Plot 1: 10 Hz Multi-Sensor Time Series (Acceleration Magnitude & PIR Motion)",
    xaxis_title="Elapsed Time (Seconds)",
    template="plotly_white",
)
fig1.update_yaxes(title_text="Acceleration Magnitude (g)", secondary_y=False)
fig1.update_yaxes(
    title_text="PIR Motion State (0/1)", secondary_y=True, range=[-0.1, 1.1]
)
fig1.write_html("plot1_time_series.html")
fig1.show()

In [5]:
idle_data = df[df["PIR_Motion"] == 0]["Accel_Mag_Smooth"]

fig2 = px.histogram(
    idle_data,
    x="Accel_Mag_Smooth",
    nbins=40,
    marginal="rug",
    title="Plot 2: Distribution of Ambient Noise in Resting State (Accel_Mag)",
    labels={"Accel_Mag_Smooth": "Acceleration Magnitude (g)"},
    opacity=0.7,
    color_discrete_sequence=["#1f77b4"],
)
fig2.update_layout(template="plotly_white", yaxis_title="Sample Count")
fig2.write_html("plot2_distribution.html")
fig2.show()

In [6]:
df["Event_ID"] = (
    (df["Force_Detected"] != df["Force_Detected"].shift()).cumsum()
)
events = (
    df[df["Force_Detected"]]
    .groupby("Event_ID")
    .agg(
        duration_s=("elapsed_seconds", lambda x: x.max() - x.min()),
        has_pir=("PIR_Motion", "max"),
    )
    .reset_index()
)
events["Event_Type"] = np.where(
    events["duration_s"] < 0.3, "fast Bump (<0.3s)", "Sustained Theft (>0.5s)"
)

fig3 = px.box(
    events,
    x="Event_Type",
    y="duration_s",
    points="all",
    color="Event_Type",
    title="Plot 3: Movement Event Duration by Category (H2 Validation)",
    labels={"duration_s": "Event Duration (Seconds)", "Event_Type": "Interaction Category"},
)
fig3.update_layout(template="plotly_white")
fig3.write_html("plot3_event_durations.html")
fig3.show()

In [7]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import confusion_matrix

# Derive Ground-Truth 'Actual_Event' Label (0 = Normal/Bump, 1 = Theft Attempt)
df["Force_Detected"] = df["Accel_Mag_Smooth"] > 0.3

# Group contiguous movement segments to measure duration
df["Segment_ID"] = (
    df["Force_Detected"] != df["Force_Detected"].shift()
).cumsum()
segment_durations = df.groupby("Segment_ID")["elapsed_seconds"].transform(
    lambda x: x.max() - x.min()
)

# Label Actual Event: Ground truth positive if duration > 0.5s AND PIR detected motion
df["Actual_Event"] = np.where(
    (segment_durations > 0.5) & (df["PIR_Motion"] == 1), 1, 0
)

# Model Prediction: Evaluates the Fused Sensor Alarm Trigger
df["Predicted_Event"] = df["Fused_Alarm_Trigger"].astype(int)

# Generate Confusion Matrix Values
labels = [0, 1]
cm = confusion_matrix(df["Actual_Event"], df["Predicted_Event"], labels=labels)

# Plot Confusion Matrix Heatmap using Plotly
fig_cm = px.imshow(
    cm,
    x=["No Alarm", "Alarm"],
    y=["Bump/low acceleration", "Theft/ high acceleration"],
    text_auto=True,
    color_continuous_scale="Blues",
    title="Confusion Matrix for Dual-Sensor",
    labels=dict(x="Alarm", y="acceleration", color="Sample Count"),
)

fig_cm.update_layout(template="plotly_white")
fig_cm.write_html("plot_confusion_matrix.html")
fig_cm.show()

## Statistical Testing & Hypothesis Validation

In [8]:
# Stationarity Test (Augmented Dickey-Fuller)
adf_result = adfuller(df["Accel_Mag_Smooth"])
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"ADF p-value: {adf_result[1]:.4e}")

ADF Statistic: -7.0809
ADF p-value: 4.6700e-10


### H1: PIR + Accelerometer Fusion Reduces False Positive Rate

False Alarm if PIR-only or Accel-only

In [12]:
pir_fpr = (
    (df["PIR_Motion"] == 1) & (~df["Force_Detected"])
).mean() * 100
accel_fpr = (
    (df["Force_Detected"]) & (df["PIR_Motion"] == 0)
).mean() * 100
fused_fpr = (
    (df["Fused_Alarm_Trigger"]) & (df["elapsed_seconds"] < 10)
).mean() * 100  # Baseline false alarm rate

print(f"\nH1 Validation (Sensor Fusion):")
print(f"PIR-Only False Positive Rate: {pir_fpr:.2f}%")
print(f"Accel-Only False Positive Rate: {accel_fpr:.2f}%")
print(f"Fused Multi-Sensor FPR: {fused_fpr:.2f}% (Target: < 5.0%)")


H1 Validation (Sensor Fusion):
PIR-Only False Positive Rate: 3.83%
Accel-Only False Positive Rate: 0.92%
Fused Multi-Sensor FPR: 0.00% (Target: < 5.0%)


H2: Movement Duration Thresholding

In [13]:
bumps = events[events["Event_Type"] == "Transient Bump (<0.3s)"]["duration_s"]
thefts = events[events["Event_Type"] == "Sustained Theft (>0.5s)"]["duration_s"]

if len(bumps) > 0 and len(thefts) > 0:
    t_stat, p_val = stats.ttest_ind(bumps, thefts, equal_var=False)
    print(f"\nH2 Validation (Duration Discrimination):")
    print(f"Two-sample Welch's t-test: t = {t_stat:.4f}, p = {p_val:.4e}")
    print(
        f"Mean Bump Duration: {bumps.mean():.3f}s | Mean Theft Duration: {thefts.mean():.3f}s"
    )